Transformer GPU 使用測試

In [ ]:
## pip install transformers torch

In [1]:
import torch
from transformers import AutoModel

In [2]:
# 檢查 GPU 是否可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 載入預訓練模型（這裡使用 BERT）
model_name = "bert-base-uncased"
model = AutoModel.from_pretrained(model_name)

# 將模型移動到 GPU
model.to(device)

# 顯示 GPU 使用狀況
if device.type == "cuda":
    allocated = torch.cuda.memory_allocated(device) / 1024**2  # 已分配記憶體 (MB)
    reserved = torch.cuda.memory_reserved(device) / 1024**2    # 預留記憶體 (MB)
    print(f"GPU Memory - Allocated: {allocated:.2f} MB, Reserved: {reserved:.2f} MB")
else:
    print("No GPU detected.")

Using device: cuda


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

GPU Memory - Allocated: 418.73 MB, Reserved: 472.00 MB


Transformer LLM 測試

In [7]:
pip show transformers

Name: transformers
Version: 4.47.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.10/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 選擇模型（7B 版本）
model_id = "codellama/CodeLlama-7b-hf"

# 檢查 GPU 是否可用，並設定設備
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 載入 tokenizer 和模型
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="cuda:0"
)

# 建立生成 pipeline
code_generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# 測試程式碼補全
prompt = "def fibonacci(n):"
output = code_generator(prompt, max_length=100, do_sample=True)

print("Generated Code:")
print(output[0]['generated_text'])

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Generated Code:
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1)+


In [ ]:
device_map = {
    "transformer.h.0": 0,  # 第一層放到 GPU 0
    "transformer.h.1": 1,  # 第二層放到 GPU 1
    "transformer.h.2": 0,  # 第三層放回 GPU 0
    "lm_head": "cpu"       # 輸出層放到 CPU
}